## 1. Lendo o arquivo fonte

Nesta prática, vamos partir da mesma página Markdown usada no tamanho fixo. A fonte ainda não é um conjunto de chunks. Primeiro extraímos o documento e depois deixamos o Docling identificar as unidades que a estrutura oferece.

In [1]:
from io import BytesIO
from pathlib import Path

from chunking_common import parse_frontmatter

DATA_PATH = Path("../data/integracoes-resilientes-webhooks.md")
source_document = DATA_PATH.read_text(encoding="utf-8")

print(f"arquivo fonte: {DATA_PATH}")
print(f"tamanho do arquivo: {len(source_document)} caracteres")

arquivo fonte: ../data/integracoes-resilientes-webhooks.md
tamanho do arquivo: 1589 caracteres


## 2. Separando metadados e texto bruto

O frontmatter descreve a origem da página inteira. Vamos preservá-lo como metadado global e enviar apenas o corpo Markdown para o Docling, para que esses campos não sejam confundidos com conteúdo do livro.

In [2]:
source_page = parse_frontmatter(source_document)
metadata = source_page.metadata

print(f"livro: {metadata.book_title}")
print(f"trecho: {metadata.chapter} > {metadata.section}")
print(f"páginas: {metadata.page_start}-{metadata.page_end}")
print(f"texto bruto: {len(source_page.text)} caracteres")

livro: Integracoes Resilientes: webhooks, filas e retentativas na pratica
trecho: Capitulo 4 - Webhooks em producao > 4.3 Timeouts e retentativas
páginas: 118-119
texto bruto: 1363 caracteres


## 3. Extraindo unidades com o HierarchicalChunker

O `DocumentConverter` transforma o Markdown em um `DoclingDocument`. O `HierarchicalChunker` usa headings, parágrafos, tabelas e código como unidades estruturais e devolve cada unidade com os metadados do documento. Ainda não estamos tentando encaixar tudo em um limite de tokens.

In [3]:
from docling.chunking import HierarchicalChunker
from docling.datamodel.base_models import DocumentStream
from docling.document_converter import DocumentConverter


CONTENT_TYPE_NAMES = {
    "text": "texto",
    "table": "tabela",
    "code": "código",
    "picture": "imagem",
}


def content_types(chunk):
    labels = [getattr(item.label, "value", str(item.label)) for item in chunk.meta.doc_items]
    return tuple(dict.fromkeys(labels))


def readable_content_types(types):
    return " + ".join(CONTENT_TYPE_NAMES.get(content_type, content_type) for content_type in types)


document_stream = DocumentStream(
    name=DATA_PATH.name,
    stream=BytesIO(source_page.text.encode("utf-8")),
)
document = DocumentConverter().convert(document_stream).document
hierarchical_chunker = HierarchicalChunker()
hierarchical_chunks = list(hierarchical_chunker.chunk(document))

print(f"documento extraído: {type(document).__name__}")
print(f"unidades estruturais: {len(hierarchical_chunks)}")
print(f"heading comum: {' > '.join(hierarchical_chunks[0].meta.headings)}")
print("unidade | conteúdo")
print("--------|-------------------")
for position, chunk in enumerate(hierarchical_chunks, start=1):
    types = readable_content_types(content_types(chunk))
    print(f"{position:>7} | {types:<17}")

documento extraído: DoclingDocument
unidades estruturais: 6
heading comum: 4.3 Timeouts e retentativas em webhooks
unidade | conteúdo
--------|-------------------
      1 | texto            
      2 | texto            
      3 | tabela           
      4 | texto            
      5 | código           
      6 | texto + código   


## 4. Preparando os candidatos estruturais

O Docling reconheceu a tabela e o payload como tipos diferentes de conteúdo. Para a tabela, usamos sua serialização Markdown. Os headings continuam nos metadados e serão recolocados uma vez na unidade final.

In [4]:
from dataclasses import dataclass


@dataclass(frozen=True)
class StructuralCandidate:
    candidate_id: str
    text: str
    heading_path: tuple[str, ...]
    content_types: tuple[str, ...]

    @property
    def readable_types(self):
        return readable_content_types(self.content_types)


def serialize_structural_chunk(chunk, document):
    types = content_types(chunk)
    if types == ("table",):
        return chunk.meta.doc_items[0].export_to_markdown(doc=document)
    return chunk.text


def to_structural_candidate(position, chunk, document):
    return StructuralCandidate(
        candidate_id=f"unit-structural-{position:02d}",
        text=serialize_structural_chunk(chunk, document),
        heading_path=tuple(chunk.meta.headings),
        content_types=content_types(chunk),
    )


candidates = [
    to_structural_candidate(position, chunk, document)
    for position, chunk in enumerate(hierarchical_chunks, start=1)
]

print(f"heading comum: {' > '.join(candidates[0].heading_path)}")
print("unidade              conteúdo          caracteres")
print("--------------------|-----------------|------------")
for candidate in candidates:
    print(f"{candidate.candidate_id:<20} {candidate.readable_types:<17} {len(candidate.text):>8}")

heading comum: 4.3 Timeouts e retentativas em webhooks
unidade              conteúdo          caracteres
--------------------|-----------------|------------
unit-structural-01   texto                  267
unit-structural-02   texto                  262
unit-structural-03   tabela                 521
unit-structural-04   texto                   91
unit-structural-05   código                 119
unit-structural-06   texto + código         232


## 5. Contando tokens com o tokenizer do modelo

Agora entra a restrição técnica. Vamos contar tokens com o tokenizer da família de embeddings que usaremos depois. O valor 220 é o orçamento desta prática: pequeno o suficiente para tornar visível a composição, mas suficiente para manter a tabela inteira.

In [5]:
from transformers import AutoTokenizer


TOKENIZER_NAME = "sentence-transformers/all-MiniLM-L6-v2"
MAX_CHUNK_TOKENS = 220
tokenizer = AutoTokenizer.from_pretrained(TOKENIZER_NAME)


def count_tokens(text):
    return len(tokenizer.encode(text, add_special_tokens=True, truncation=False))


print(f"tokenizer: {TOKENIZER_NAME}")
print(f"orçamento por chunk final: {MAX_CHUNK_TOKENS} tokens")
print("unidade              conteúdo          tokens do conteúdo")
print("--------------------|-----------------|-------------------")
for candidate in candidates:
    print(f"{candidate.candidate_id:<20} {candidate.readable_types:<17} {count_tokens(candidate.text):>8}")

tokenizer: sentence-transformers/all-MiniLM-L6-v2
orçamento por chunk final: 220 tokens
unidade              conteúdo          tokens do conteúdo
--------------------|-----------------|-------------------
unit-structural-01   texto                   94
unit-structural-02   texto                   91
unit-structural-03   tabela                 199
unit-structural-04   texto                   31
unit-structural-05   código                  65
unit-structural-06   texto + código          90


## 6. Compondo chunks dentro do orçamento

A política fica explícita no código. Percorremos as unidades na ordem original e agrupamos apenas vizinhas com a mesma hierarquia. Antes de adicionar uma unidade, contamos o texto final com o heading e verificamos se ele continua dentro do orçamento.

In [6]:
@dataclass(frozen=True)
class StructuralChunk:
    chunk_id: str
    text: str
    candidate_ids: tuple[str, ...]
    heading_path: tuple[str, ...]
    content_types: tuple[str, ...]
    token_count: int

    @property
    def readable_types(self):
        return readable_content_types(self.content_types)


def render_candidate_group(group):
    heading = "\n".join(group[0].heading_path)
    body = "\n\n".join(candidate.text for candidate in group)
    return f"{heading}\n\n{body}" if heading else body


def unique_content_types(group):
    return tuple(dict.fromkeys(
        content_type
        for candidate in group
        for content_type in candidate.content_types
    ))


def build_structural_chunk(order, group):
    text = render_candidate_group(group)
    return StructuralChunk(
        chunk_id=f"chunk-structural-{order:02d}",
        text=text,
        candidate_ids=tuple(candidate.candidate_id for candidate in group),
        heading_path=group[0].heading_path,
        content_types=unique_content_types(group),
        token_count=count_tokens(text),
    )


def compose_structural_chunks(candidates, max_tokens):
    chunks = []
    current = []

    for candidate in candidates:
        single_candidate = build_structural_chunk(len(chunks) + 1, [candidate])
        if single_candidate.token_count > max_tokens:
            raise ValueError(
                f"{candidate.candidate_id} excede o orçamento sozinho, "
                "esse elemento precisa de uma estratégia própria de divisão."
            )

        if current:
            same_heading = candidate.heading_path == current[0].heading_path
            combined = build_structural_chunk(len(chunks) + 1, current + [candidate])
            if not same_heading or combined.token_count > max_tokens:
                chunks.append(build_structural_chunk(len(chunks) + 1, current))
                current = []

        current.append(candidate)

    if current:
        chunks.append(build_structural_chunk(len(chunks) + 1, current))

    return tuple(chunks)


structural_chunks = compose_structural_chunks(candidates, MAX_CHUNK_TOKENS)

print("composição das unidades:")
for chunk in structural_chunks:
    units = " + ".join(candidate_id.rsplit("-", 1)[-1] for candidate_id in chunk.candidate_ids)
    chunk_number = chunk.chunk_id.rsplit("-", 1)[-1]
    print(f"unidades {units} -> chunk final {chunk_number} | {chunk.token_count} tokens | {chunk.readable_types}")

composição das unidades:
unidades 01 + 02 -> chunk final 01 | 199 tokens | texto
unidades 03 -> chunk final 02 | 215 tokens | tabela
unidades 04 + 05 + 06 -> chunk final 03 | 198 tokens | texto + código


## 7. Inspecionando as unidades recuperáveis

A estrutura orientou os candidatos. O tokenizer controlou a composição. A tabela permanece como tabela, o código permanece como código e os chunks finais carregam o heading apenas uma vez.

In [7]:
from textwrap import fill


def format_chunk_for_display(text, width=72):
    formatted_lines = []
    in_code_block = False

    for line in text.splitlines():
        if line.startswith("```"):
            in_code_block = not in_code_block
            formatted_lines.append(line)
        elif in_code_block or line.startswith("|"):
            formatted_lines.append(line)
        elif line.strip():
            formatted_lines.extend(fill(line, width=width).splitlines())
        else:
            formatted_lines.append("")

    return "\n".join(formatted_lines)


for chunk in structural_chunks:
    print("=" * 72)
    print(f"{chunk.chunk_id} | {chunk.token_count} tokens | {chunk.readable_types}")
    print(format_chunk_for_display(chunk.text))
print("=" * 72)

chunk-structural-01 | 199 tokens | texto
4.3 Timeouts e retentativas em webhooks

Quando um provedor envia um webhook, ele espera uma resposta rapida do
consumidor. Se a conexao expira antes de receber confirmacao, o provedor
nao sabe se o evento falhou antes de chegar, se foi processado
parcialmente ou se a resposta se perdeu no caminho de volta.

Por isso, timeout nao deve ser tratado como erro definitivo. Em geral,
ele entra na mesma familia de falhas temporarias: o provedor registra a
tentativa, agenda uma nova entrega e preserva o mesmo identificador de
evento para permitir deduplicacao no consumidor.
chunk-structural-02 | 215 tokens | tabela
4.3 Timeouts e retentativas em webhooks

|   status | significado                               | acao recomendada            |
|----------|-------------------------------------------|-----------------------------|
|      200 | evento recebido e persistido              | encerrar entrega            |
|      408 | consumidor nao respondeu dent

## 8. Anexando metadados

A unidade final carrega os metadados globais do frontmatter e os dados locais produzidos durante o chunking. Isso permite filtrar a origem, reconstruir a posição e saber que tipos de conteúdo foram agrupados.

In [8]:
metadata_chunks = []
for order, chunk in enumerate(structural_chunks, start=1):
    metadata_chunks.append({
        "text": chunk.text,
        "metadata": {
            "book_title": metadata.book_title,
            "edition": metadata.edition,
            "chapter": metadata.chapter,
            "section": metadata.section,
            "page_start": metadata.page_start,
            "page_end": metadata.page_end,
            "chunk_id": chunk.chunk_id,
            "strategy": "hierarchical_then_token_aware",
            "order": order,
            "heading_path": list(chunk.heading_path),
            "content_types": list(chunk.content_types),
            "token_count": chunk.token_count,
            "source_file": str(DATA_PATH),
        },
    })

first_metadata = metadata_chunks[0]["metadata"]
print("metadados globais:")
print(f"  livro: {first_metadata['book_title']}")
print(f"  edição: {first_metadata['edition']}")
print(f"  páginas: {first_metadata['page_start']}-{first_metadata['page_end']}")
print("\namostra do primeiro chunk:")
for key in ["chunk_id", "strategy", "order", "heading_path", "content_types", "token_count", "source_file"]:
    value = first_metadata[key]
    if key == "heading_path":
        value = " > ".join(value)
    elif key == "content_types":
        value = ", ".join(value)
    print(f"  {key}: {value}")

metadados globais:
  livro: Integracoes Resilientes: webhooks, filas e retentativas na pratica
  edição: 2a edicao
  páginas: 118-119

amostra do primeiro chunk:
  chunk_id: chunk-structural-01
  strategy: hierarchical_then_token_aware
  order: 1
  heading_path: 4.3 Timeouts e retentativas em webhooks
  content_types: text
  token_count: 199
  source_file: ../data/integracoes-resilientes-webhooks.md


## 9. Recuperando candidatos

A busca permanece simples de propósito. Ela conta termos da query nos chunks já formados. O ponto é observar qual unidade estrutural chega ao retrieval, sem transformar a prática em uma aula de ranking ou de banco vetorial.

In [9]:
from chunking_common import print_search_summary, search


query = "como lidar com timeout em webhooks?"
results = search(query, metadata_chunks)
print_search_summary(query, results)

query: como lidar com timeout em webhooks?
resultados por contagem simples de termos:
posição | chunk                  | score | termos
--------|------------------------|-------|----------------
      1 | chunk-structural-01    |     2 | timeout, webhooks
      2 | chunk-structural-03    |     2 | timeout, webhooks
      3 | chunk-structural-02    |     1 | webhooks

maior score simplificado: 2
candidato(s) no topo: chunk-structural-01, chunk-structural-03
prévia do primeiro candidato no topo:
4.3 Timeouts e retentativas em webhooks Quando um provedor envia um
webhook, ele espera uma resposta rapida do consumidor. Se a conexao
expira antes de receber confirmacao, o prove...


O resultado mostra as duas decisões separadas. O `HierarchicalChunker` encontrou as unidades estruturais e trouxe seus metadados. Nossa composição aplicou o orçamento do tokenizer para formar chunks finais. O `HybridChunker` automatiza uma combinação parecida, mas deixá-la explícita aqui torna a política observável. Os embeddings ficam para a prática semântica.